# Lecture 7 · CNN · watching shapes flow through

**Goal** · build a small CNN, print tensor shapes at every layer, compute receptive field by hand.

Shape bookkeeping is the single most useful skill when debugging a vision model. We'll make it explicit.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1 · Conv output size formula · let's verify

$$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

In [ ]:
def out_size(W, K, P, S):
    return (W - K + 2*P) // S + 1

# stride 1 · padding 1 · 3x3 kernel · preserves size
print(f'32  → {out_size(32, 3, 1, 1)}  (same)')
# stride 2 · halves
print(f'32  → {out_size(32, 3, 1, 2)}  (half)')
# 7x7 stride-2 stem · 224 → 112
print(f'224 → {out_size(224, 7, 3, 2)}  (ResNet stem)')

## 2 · Build a tiny CIFAR-10 CNN · print shapes

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # input 3 × 32 × 32
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)       # 32 × 32 × 32
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)      # 64 × 32 × 32
        self.pool1 = nn.MaxPool2d(2)                       # 64 × 16 × 16
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)     # 128 × 16 × 16
        self.pool2 = nn.MaxPool2d(2)                       # 128 × 8 × 8
        self.avgpool = nn.AdaptiveAvgPool2d(1)             # 128 × 1 × 1
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x, verbose=False):
        def p(tag, t):
            if verbose:
                print(f'{tag:10s} {tuple(t.shape)}')
        p('input', x)
        x = F.relu(self.conv1(x)); p('conv1', x)
        x = F.relu(self.conv2(x)); p('conv2', x)
        x = self.pool1(x); p('pool1', x)
        x = F.relu(self.conv3(x)); p('conv3', x)
        x = self.pool2(x); p('pool2', x)
        x = self.avgpool(x); p('avgpool', x)
        x = x.flatten(1); p('flatten', x)
        x = self.fc(x); p('fc', x)
        return x

model = TinyCNN()
x = torch.zeros(2, 3, 32, 32)
_ = model(x, verbose=True)
print(f'\nparams · {sum(p.numel() for p in model.parameters()):,}')

## 3 · Receptive field · compute by layer

Formula · `RF_L = RF_{L-1} + (K_L - 1) · product(strides up to L-1)`

In [ ]:
layers = [
    ('conv1', 3, 1),
    ('conv2', 3, 1),
    ('pool1', 2, 2),
    ('conv3', 3, 1),
    ('pool2', 2, 2),
]

rf = 1
jump = 1
print(f'{"layer":10s} {"K":>3} {"S":>3} {"jump":>5} {"RF":>4}')
for name, K, S in layers:
    rf = rf + (K - 1) * jump
    jump = jump * S
    print(f'{name:10s} {K:3d} {S:3d} {jump:5d} {rf:4d}')

**Read** · by pool2 (layer 5) every feature depends on an 11×11 patch of the input. With 32×32 images that's already a third of the image.

Add more layers → receptive field grows → the network sees more of the image in each feature.

This is the key quantity behind "why deeper networks understand objects, not just edges."

## 4 · 1×1 conv · channel mixing in action

In [ ]:
# 256 channels -> 64 channels via 1x1
reduce = nn.Conv2d(256, 64, kernel_size=1)
print(f'1x1 conv 256 -> 64 params · {sum(p.numel() for p in reduce.parameters()):,}')

# Compare to a 3x3 doing the same channel reduction
full = nn.Conv2d(256, 64, kernel_size=3, padding=1)
print(f'3x3 conv 256 -> 64 params · {sum(p.numel() for p in full.parameters()):,}')

print('\n1x1 is 9x cheaper AND preserves spatial structure. Used everywhere.')